<a href="https://colab.research.google.com/github/minhanaseer/MiniOS/blob/main/proposaltestng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -q https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
!wget -q https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv
!wget -q https://physionet.org/files/ptb-xl-plus/1.0.1/labels/12sl_statements.csv
!ls -lh *.csv

^C
-rw-r--r-- 1 root root 4.6M Oct 28 15:55 12sl_statements.csv
-rw-r--r-- 1 root root 6.3M Nov  3  2022 ptbxl_database.csv
-rw-r--r-- 1 root root 9.5K Apr 23  2020 scp_statements.csv


In [2]:
import pandas as pd, ast

db = pd.read_csv('ptbxl_database.csv', index_col='ecg_id')
scp = pd.read_csv('scp_statements.csv', index_col=0)

mi_codes = set(scp.index[scp['diagnostic_class'] == 'MI'])
db['mi'] = db['scp_codes'].apply(lambda s: any(c in mi_codes for c in ast.literal_eval(s)))
db['sex_label'] = db['sex'].map({0: 'Male', 1: 'Female'})
db['age_band'] = db['age'].apply(lambda a: 'Unknown' if pd.isna(a) else ('Under 65' if a < 65 else '65 and over'))

print('Total ECGs:', len(db))
print('ECGs with MI:', db['mi'].sum())
print('MI codes used:', sorted(mi_codes))

Total ECGs: 21799
ECGs with MI: 5469
MI codes used: ['ALMI', 'AMI', 'ASMI', 'ILMI', 'IMI', 'INJAL', 'INJAS', 'INJIL', 'INJIN', 'INJLA', 'IPLMI', 'IPMI', 'LMI', 'PMI']


In [3]:
mi = db[db['mi']]
print(pd.crosstab(mi['age_band'], mi['sex_label'], margins=True, margins_name='Total'))


sex_label    Female  Male  Total
age_band                        
65 and over    1516  1677   3193
Under 65        546  1730   2276
Total          2062  3407   5469


In [4]:
sl = pd.read_csv('12sl_statements.csv')
print(sl.columns.tolist())
print(sl.head(10))

ParserError: Error tokenizing data. C error: EOF inside string starting at row 6097

In [5]:
!ls -lh 12sl_statements.csv
!head -c 800 12sl_statements.csv

-rw-r--r-- 1 root root 4.6M Oct 28 15:55 12sl_statements.csv
ecg_id,statements,statements_cat,statements_ext,statements_ext_snomed
1,"['NSR', 'NML']","['NSR', 'NML']","[('NSR', 100.0), ('NML', 100.0)]","[(441840, 100.0), (4008453, 100.0), (4010019, 100.0), (4021775, 100.0), (4023995, 100.0), (4024571, 100.0), (4042140, 100.0), (4065279, 100.0), (4091898, 100.0), (4094294, 100.0), (4103183, 100.0), (4114162, 100.0), (4115390, 100.0), (4117134, 100.0), (4117930, 100.0), (4142265, 100.0), (4145513, 100.0), (4185503, 100.0), (4199402, 100.0), (4227253, 100.0), (4276669, 100.0), (40480457, 100.0), (43531057, 100.0)]"
21803,"['SRTH', 'MSAR', 'RBBB', 'IT', 'AB']","['SRTH;MSAR', 'RBBB', 'IT', 'AB']","[('SRTH', 100.0), ('MSAR', 100.0), ('RBBB', 100.0), ('IT', 100.0), ('AB', 100.0)]","[(134057, 100.0), (313791, 100.0), (314059, 100.0), (316999, 100.0), (32042

In [6]:
sl = pd.read_csv('12sl_statements.csv', engine='python', on_bad_lines='warn')
print('Rows read:', len(sl))

sl['codes'] = sl['statements'].apply(ast.literal_eval)
counts = sl.explode('codes')['codes'].value_counts()

print('\nNumber of different 12SL statements:', len(counts))
print('\nStatements containing "MI":')
print(counts[counts.index.str.contains('MI', na=False)])


Rows read: 6096

Number of different 12SL statements: 154

Statements containing "MI":
codes
IMI       517
SMI       186
AMI       156
ASMI      122
STEMI      63
LMI        44
ALMI       35
IPMI       13
QESPMI      3
POSTMI      3
MILT        2
MIT         1
Name: count, dtype: int64


/tmp/ipykernel_1072/1935451681.py:1: ParserWarning: Skipping line 6098: unexpected end of data

  sl = pd.read_csv('12sl_statements.csv', engine='python', on_bad_lines='warn')


In [7]:
has_12sl = mi.index.isin(sl['ecg_id'])
print(pd.crosstab(mi['age_band'], has_12sl, margins=True, margins_name='Total'))

col_0        False  True  Total
age_band                       
65 and over   2378   815   3193
Under 65      1840   436   2276
Total         4218  1251   5469


In [8]:
!wc -l 12sl_statements.csv
!tail -c 400 12sl_statements.csv

6097 12sl_statements.csv
0.0), (4199402, 100.0), (4215406, 100.0), (4227253, 100.0), (4228448, 100.0), (4248028, 100.0), (4274025, 100.0), (4277352, 100.0), (40480457, 100.0), (40484102, 100.0), (43531056, 100.0), (43531057, 100.0), (44784217, 100.0)]"
6102,"['NSR', 'LVH3', 'LPAREN', 'RAVL', 'COMMA', 'CORNPROD', 'RPAREN', 'BORDE']","['NSR', 'LVH3', 'RAVL', 'CORNPROD', 'BORDE']","[('NSR', 100.0), ('BORDE', 100.0)]","[(4418

In [9]:
!rm -f 12sl_statements.csv
!wget -c --tries=5 https://physionet.org/files/ptb-xl-plus/1.0.1/labels/12sl_statements.csv

--2026-10-28 16:11:48--  https://physionet.org/files/ptb-xl-plus/1.0.1/labels/12sl_statements.csv
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17764014 (17M) [text/plain]
Saving to: ‘12sl_statements.csv’

12sl_statements.csv 100%[===================>]  16.94M  26.9KB/s    in 9m 26s  

2026-10-28 16:21:16 (30.7 KB/s) - ‘12sl_statements.csv’ saved [17764014/17764014]



In [10]:
!ls -lh 12sl_statements.csv
!wc -l 12sl_statements.csv
!tail -c 150 12sl_statements.csv

-rw-r--r-- 1 root root 17M Nov 16  2022 12sl_statements.csv
21800 12sl_statements.csv
4274025, 100.0), (4277352, 100.0), (37017194, 100.0), (40480457, 100.0), (40484102, 100.0), (43531056, 100.0), (43531057, 100.0), (44784217, 100.0)]"


In [11]:
sl = pd.read_csv('12sl_statements.csv')
print('Rows read:', len(sl))

sl['codes'] = sl['statements'].apply(ast.literal_eval)
counts = sl.explode('codes')['codes'].value_counts()
print(counts[counts.index.str.contains('MI', na=False)])

has_12sl = mi.index.isin(sl['ecg_id'])
print(pd.crosstab([mi['age_band'], mi['sex_label']], has_12sl, margins=True, margins_name='Total'))

Rows read: 21799
codes
IMI       2305
SMI        708
AMI        629
ASMI       605
LMI        192
STEMI      178
ALMI       163
IPMI        61
QESPMI      31
MILT        13
POSTMI       9
MIT          1
Name: count, dtype: int64
col_0                  True  Total
age_band    sex_label             
65 and over Female     1516   1516
            Male       1677   1677
Under 65    Female      546    546
            Male       1730   1730
Total                  5469   5469
